# Diffusion 大一统

推荐你读 https://arxiv.org/abs/2011.13456 Score-Based Generative Modeling through Stochastic Differential Equations 这篇论文完整揭示了我们所说的大一统。

### 回顾

回顾 DSM 的基本逻辑。我们研究一个加了噪声的分布 $p_\sigma(\tilde{\mathbf{x}})$ $$p_\sigma(\tilde{\mathbf{x}}) = \int p_{data}(\mathbf{x}) \underbrace{p_\sigma(\tilde{\mathbf{x}}|\mathbf{x})}_{\text{加噪过程}} d\mathbf{x}$$
其中 $p_\sigma(\tilde{\mathbf{x}}|\mathbf{x}) = \mathcal{N}(\tilde{\mathbf{x}} | \mathbf{x}, \sigma^2 \mathbf{I})$ 被称为扰动核

目标权重 $$\boldsymbol{\theta}^* = \arg \min_{\boldsymbol{\theta}} \sum_{i=1}^N \sigma_i^2 \mathbb{E}_{p_{\text{data}}(\mathbf{x})} \mathbb{E}_{p_{\sigma_i}(\tilde{\mathbf{x}} | \mathbf{x})} [\| \mathbf{s}_{\boldsymbol{\theta}}(\tilde{\mathbf{x}}, \sigma_i) - \nabla_{\tilde{\mathbf{x}}} \log p_{\sigma_i}(\tilde{\mathbf{x}} | \mathbf{x}) \|_2^2]$$
其中 $\mathbf{s}_{\boldsymbol{\theta}}$ 是神经网络预测的 $\nabla \log p(\tilde{\mathbf{x}})$。解释一下 $argmin$ 这个符号，简而言之就是使目标最小的 $\theta$。

更具体的，$\nabla_{\tilde{\mathbf{x}}} \log p_{\sigma_i}(\tilde{\mathbf{x}}|\mathbf{x}) = \frac{\mathbf{x} - \tilde{\mathbf{x}}}{\sigma_i^2}$，代入得到 $$\boldsymbol{\theta}^* = \arg \min_{\boldsymbol{\theta}} \sum_{i=1}^N \sigma_i^2 \mathbb{E}_{\mathbf{x} \sim p_{\text{data}}, \tilde{\mathbf{x}} \sim \mathcal{N}(\mathbf{x}, \sigma_i^2\mathbf{I})} \left[ \left\| \mathbf{s}_{\boldsymbol{\theta}}(\tilde{\mathbf{x}}, \sigma_i) - \frac{\mathbf{x} - \tilde{\mathbf{x}}}{\sigma_i^2} \right\|_2^2 \right]$$

推理 $$\mathbf{x}_i^m = \mathbf{x}_i^{m-1} + \epsilon_i \mathbf{s}_{\boldsymbol{\theta}^*}(\mathbf{x}_i^{m-1}, \sigma_i) + \sqrt{2\epsilon_i}\mathbf{z}_i^m$$
其中 $\epsilon_i \mathbf{s}_{\boldsymbol{\theta}}$ 推向高概率区，$\sqrt{2\epsilon_i}\mathbf{z}_i^m$ 加上一点随机抖动，防止卡在局部最优，或者说 Langevin Dynamics。

你会注意到我们现在 DSM 的记号与第一章很不一样，但是实际上他们指的是同一件事。我们为了和 Score-Based Generative Modeling through Stochastic Differential Equations 原文对齐而采用原文中的记号。

接下来回顾 DDIM 的基本逻辑。我们刚刚在上一章讲述过。

前向加噪 $$p(\mathbf{x}_i | \mathbf{x}_{i-1}) = \mathcal{N}(\mathbf{x}_i; \sqrt{1 - \beta_i} \mathbf{x}_{i-1}, \beta_i \mathbf{I})$$

边缘分布 $$p_{\alpha_i}(\mathbf{x}_i | \mathbf{x}_0) = \mathcal{N}(\mathbf{x}_i; \sqrt{\alpha_i} \mathbf{x}_0, (1 - \alpha_i) \mathbf{I})$$

目标权重 $$\boldsymbol{\theta}^* = \arg \min_{\boldsymbol{\theta}} \sum_{i=1}^N (1 - \alpha_i) \mathbb{E}_{\mathbf{x} \sim p_{\text{data}}, \tilde{\mathbf{x}} \sim \mathcal{N}(\sqrt{\alpha_i}\mathbf{x}, (1-\alpha_i)\mathbf{I})} \left[ \left\| \mathbf{s}_{\boldsymbol{\theta}}(\tilde{\mathbf{x}}, i) - \frac{\sqrt{\alpha_i}\mathbf{x} - \tilde{\mathbf{x}}}{1-\alpha_i} \right\|_2^2 \right]$$

这里为你解释为什么目标权重是这样的。实际上我们上一章中推导的目标函数结果是$$L_\gamma = \sum_{t=1}^T \gamma_t \mathbb{E}_{\mathbf{x}_0 \sim q(\mathbf{x}_0), \boldsymbol{\epsilon}_t \sim \mathcal{N}(\mathbf{0}, \mathbf{I})} [\|\boldsymbol{\epsilon}_t - \boldsymbol{\epsilon}_\theta^{(t)}(\sqrt{\alpha_t} \mathbf{x}_0 + \sqrt{1 - \alpha_t} \boldsymbol{\epsilon}_t, t)\|^2]$$

我们做代换。根据高斯分布 $\mathcal{N}(\mathbf{x}; \boldsymbol{\mu}, \sigma^2\mathbf{I})$ 的性质，其对数密度的梯度（Score）为$$\nabla_{\mathbf{x}} \log p(\mathbf{x}) = -\frac{\mathbf{x} - \boldsymbol{\mu}}{\sigma^2}$$
在这里是
$$\nabla_{\mathbf{x}_t} \log p(\mathbf{x}_t | \mathbf{x}_0) = -\frac{\mathbf{x}_t - \sqrt{\alpha_t}\mathbf{x}_0}{1 - \alpha_t}$$
根据 $\mathbf{x}_t = \sqrt{\alpha_t}\mathbf{x}_0 + \sqrt{1 - \alpha_t} \boldsymbol{\epsilon}$，有 $$\nabla_{\mathbf{x}_t} \log p(\mathbf{x}_t | \mathbf{x}_0) = -\frac{\sqrt{1 - \alpha_t} \boldsymbol{\epsilon}}{1 - \alpha_t} = -\frac{\boldsymbol{\epsilon}}{\sqrt{1 - \alpha_t}}$$
直接令 $$\mathbf{s}_\theta(\mathbf{x}_t, t) = -\frac{\boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)}{\sqrt{1 - \alpha_t}}$$

代入 $$J = \sum_{t=1}^T (1 - \alpha_t) \mathbb{E} \left[ \left\| \mathbf{s}_\theta - \nabla \log p \right\|^2 \right]$$
得到 $$J = \sum_{t=1}^T \mathbb{E} [ \| \boldsymbol{\epsilon} - \boldsymbol{\epsilon}_\theta \|^2 ]$$

先到这里。SMLD 说生成图像就是沿着梯度场（Score）移动，DDPM 说生成图像就是逆转加噪过程。